# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule: Pages with declining traffic should be prioritized for review.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [27]:
!wget https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv

import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')



df.info()


--2026-08-19 01:09:21--  https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘content_refresh_anonymized.csv.4’

content_refresh_ano 100%[===================>]   6.42M  --.-KB/s    in 0.1s    

2026-08-19 01:09:21 (51.4 MB/s) - ‘content_refresh_anonymized.csv.4’ saved [6727670/6727670]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 

In [28]:
df["trend_direction"].unique()

array(['down', 'stable', 'new', 'up', 'flat'], dtype=object)

In [29]:
df["down_trend"] = df["trend_direction"] == "down"

In [35]:
df["impressions_90d"].mean()

np.float64(15132.6)

In [36]:
df["score"] = (
    df["down_trend"].astype(int) * 2
    + (df["impressions_90d"] >= 15000).astype(int)
)

df["score"]

,score
24,3
0,2
29983,2
22,2
20,2
29979,2
29978,3
18,2
8,3
5,2


In [37]:
df = df.sort_values("score", ascending=False).head(20)
df[["content_id","score","impressions_90d","trend_direction"]]

,content_id,score,impressions_90d,trend_direction
24,content_0e23e310d404,3,29541,down
4,content_d99b7a2d90ca,3,19140,down
8,content_5e6c160719bc,3,32574,down
29978,content_3dc420aa9809,3,22425,down
62,content_ea379287633b,3,38542,down
57,content_8180f9c9cbf0,3,73667,down
1,content_a1fb4e703a9e,3,15320,down
0,content_304f48230142,2,3803,down
22,content_3fb46bec4413,2,2311,down
29983,content_6880eb215048,2,2845,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Actions that could be taken: refresh, expansion, protection, pruning or monitoring



In [38]:
import numpy as np

df["action"] = np.select(
    [
        df["score"] == 3,
        df["score"] == 2,
        df["score"] == 1,
        df["score"] == 0
    ],
    [
        "Refresh",
        "Monitoring",
        "Protection",
        "Pruning"
    ],
    default="Monitoring"
)

df[["content_id","score","impressions_90d","trend_direction","action"]]


,content_id,score,impressions_90d,trend_direction,action
24,content_0e23e310d404,3,29541,down,Refresh
4,content_d99b7a2d90ca,3,19140,down,Refresh
8,content_5e6c160719bc,3,32574,down,Refresh
29978,content_3dc420aa9809,3,22425,down,Refresh
62,content_ea379287633b,3,38542,down,Refresh
57,content_8180f9c9cbf0,3,73667,down,Refresh
1,content_a1fb4e703a9e,3,15320,down,Refresh
0,content_304f48230142,2,3803,down,Monitoring
22,content_3fb46bec4413,2,2311,down,Monitoring
29983,content_6880eb215048,2,2845,down,Monitoring


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [39]:
df[["content_id","score","impressions_last_30d","impressions_90d","trend_direction","action"]]

,content_id,score,impressions_last_30d,impressions_90d,trend_direction,action
24,content_0e23e310d404,3,9313,29541,down,Refresh
4,content_d99b7a2d90ca,3,4211,19140,down,Refresh
8,content_5e6c160719bc,3,5696,32574,down,Refresh
29978,content_3dc420aa9809,3,1686,22425,down,Refresh
62,content_ea379287633b,3,4257,38542,down,Refresh
57,content_8180f9c9cbf0,3,10616,73667,down,Refresh
1,content_a1fb4e703a9e,3,2501,15320,down,Refresh
0,content_304f48230142,2,578,3803,down,Monitoring
22,content_3fb46bec4413,2,292,2311,down,Monitoring
29983,content_6880eb215048,2,1070,2845,down,Monitoring


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.